# PMBB v4 hearing impairment and tinnitus phenotypes

This final workflow creates **separate** hearing-impairment and tinnitus phenotypes from PMBB release 4.0.

PMBB v4 stores the relevant evidence in two OMOP tables:

- **Hearing impairment:** ICD-9-CM/ICD-10-CM condition events mapped by PMBB to PhecodeX `SO_396*`
- **Standard tinnitus:** ICD-9-CM `388.3*` and ICD-10-CM `H93.1*` events stored in `observation`
- **Pulsatile tinnitus:** ICD-10-CM `H93.A*` condition events mapped to `SO_397`, `SO_397.1`, and `SO_397.11`

Case/control rules follow the PMBB approach used by Park et al.:

- **Case:** qualifying phenotype evidence on at least two distinct dates
- **Excluded one date:** qualifying evidence on exactly one date
- **Excluded related auditory phenotype:** not a case, but has a related PhecodeX ear-family event
- **Control:** no target evidence and no related auditory evidence

Hearing impairment and tinnitus are analyzed independently. Tinnitus cases may also have hearing loss. No audiogram or other auditory measurement is used because those measurements are unavailable in this PMBB v4 release.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

BASE = Path("/static/PMBB/PMBB-Release-2026-4.0/Phenotype/4.0")
PREFIX = "PMBB-Release-2026-4.0_phenotype_"

PHECODEX_FILE = BASE / f"{PREFIX}conditions_phecode_x.txt"
DX_FILE = BASE / f"{PREFIX}condition_occurrence.txt"
OBS_FILE = BASE / f"{PREFIX}observation.txt"
CONCEPT_FILE = BASE / f"{PREFIX}concept.txt"
COV_FILE = BASE / f"{PREFIX}covariates.txt"
FAM_FILE = Path(
    "/static/PMBB/PMBB-Release-2026-4.0/Imputed/common_snps_LD_pruned/"
    "PMBB-Release-2026-4.0_genetic_imputed.commonsnps.ldpruned.ALL.fam"
)

OUT = Path(
    "/project/hall/analysis/hearing-loss-genomics/analysis/elena/"
    "rarevariantExWAS/PMBBv4_phecodex"
)
OUT.mkdir(parents=True, exist_ok=True)

for path in [PHECODEX_FILE, DX_FILE, OBS_FILE, CONCEPT_FILE, COV_FILE, FAM_FILE]:
    if not path.exists():
        raise FileNotFoundError(path)

TINNITUS_ICD10_STANDARD = {"H93.1", "H93.11", "H93.12", "H93.13", "H93.19"}
TINNITUS_ICD10_PULSATILE = {"H93.A", "H93.A1", "H93.A2", "H93.A3", "H93.A9"}
TINNITUS_ICD9 = {"388.3", "388.30", "388.31", "388.32"}
TINNITUS_OBSERVATION_CODES = TINNITUS_ICD10_STANDARD | TINNITUS_ICD9

TARGETS = {
    "hearing_impairment": "SO_396",
    "tinnitus": "SO_397",
}

print("Output:", OUT)

Output: /project/hall/analysis/hearing-loss-genomics/analysis/elena/rarevariantExWAS/PMBBv4_phecodex


## 1. Load only the PhecodeX ear family

Filtering to `SO_390`–`SO_399` before joining the large condition table substantially reduces runtime. Multiple hierarchical PhecodeX rows belonging to the same `condition_occurrence_id` are collapsed into one event with a semicolon-separated audit field.


In [2]:
phe_parts = []

for chunk_number, chunk in enumerate(
    pd.read_csv(
        PHECODEX_FILE,
        sep="\t",
        usecols=[
            "condition_occurrence_id",
            "person_id",
            "condition_start_date",
            "condition_source_value",
        ],
        dtype="string",
        chunksize=2_000_000,
        low_memory=False,
    ),
    start=1,
):
    phecode = chunk["condition_source_value"].fillna("").str.strip().str.upper()
    keep = phecode.str.match(r"^SO_39[0-9](?:\.|$)", na=False)
    if keep.any():
        selected = chunk.loc[keep].copy()
        selected["phecode_x"] = phecode.loc[keep]
        phe_parts.append(selected.drop(columns="condition_source_value"))

ear_phe_rows = pd.concat(phe_parts, ignore_index=True)
ear_phe_rows["person_id"] = ear_phe_rows["person_id"].str.strip()
ear_phe_rows["condition_start_date"] = pd.to_datetime(
    ear_phe_rows["condition_start_date"], errors="coerce"
).dt.date

ear_phe = (
    ear_phe_rows
    .groupby(
        ["condition_occurrence_id", "person_id", "condition_start_date"],
        dropna=False,
        sort=False,
    )["phecode_x"]
    .agg(lambda x: ";".join(sorted(x.dropna().unique())))
    .rename("mapped_phecodes")
    .reset_index()
)

ear_phe["is_hearing_impairment"] = ear_phe["mapped_phecodes"].str.contains(
    r"(?:^|;)SO_396(?:\.|;|$)", regex=True, na=False
)
ear_phe["is_pulsatile_tinnitus"] = ear_phe["mapped_phecodes"].str.contains(
    r"(?:^|;)SO_397\.1(?:\.|;|$)", regex=True, na=False
)

print("Ear-family PhecodeX rows:", f"{len(ear_phe_rows):,}")
print("Unique ear-family clinical events:", f"{len(ear_phe):,}")
print("Hearing events:", f"{ear_phe['is_hearing_impairment'].sum():,}")
print("Pulsatile-tinnitus events:", f"{ear_phe['is_pulsatile_tinnitus'].sum():,}")

Ear-family PhecodeX rows: 655,946
Unique ear-family clinical events: 292,848
Hearing events: 134,917
Pulsatile-tinnitus events: 3,016


## 2. Recover original ICD codes for the ear-family condition events

Only condition records already identified by PhecodeX are retained during the large-file scan.


In [3]:
ear_event_ids = set(ear_phe["condition_occurrence_id"].dropna())
condition_parts = []

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DX_FILE,
        sep="\t",
        usecols=[
            "condition_occurrence_id",
            "condition_source_value",
            "condition_source_concept_id",
        ],
        dtype="string",
        chunksize=2_000_000,
        low_memory=False,
    ),
    start=1,
):
    keep = chunk["condition_occurrence_id"].isin(ear_event_ids)
    if keep.any():
        condition_parts.append(chunk.loc[keep].copy())
    if chunk_number % 5 == 0:
        print(f"Condition chunks processed: {chunk_number}")

condition_icd = pd.concat(condition_parts, ignore_index=True)
condition_icd = condition_icd.drop_duplicates("condition_occurrence_id")
condition_icd = condition_icd.rename(columns={
    "condition_source_value": "icd_code",
    "condition_source_concept_id": "source_concept_id",
})
condition_icd["icd_code"] = condition_icd["icd_code"].str.strip().str.upper()
condition_icd["source_concept_id"] = pd.to_numeric(
    condition_icd["source_concept_id"], errors="coerce"
).astype("Int64")

condition_events = ear_phe.merge(
    condition_icd,
    on="condition_occurrence_id",
    how="left",
    validate="one_to_one",
)

print("Condition events recovered:", f"{condition_events['icd_code'].notna().sum():,}")

Condition chunks processed: 5
Condition chunks processed: 10
Condition chunks processed: 15
Condition chunks processed: 20
Condition events recovered: 292,848


## 3. Extract standard tinnitus from `observation`

PMBB v4 relocated standard tinnitus records to the OMOP observation table. Exact code membership avoids including unrelated `H93` diagnoses.


In [4]:
observation_parts = []

for chunk_number, chunk in enumerate(
    pd.read_csv(
        OBS_FILE,
        sep="\t",
        usecols=[
            "observation_id",
            "person_id",
            "observation_date",
            "observation_source_value",
            "observation_source_concept_id",
        ],
        dtype="string",
        chunksize=2_000_000,
        low_memory=False,
    ),
    start=1,
):
    code_value = chunk["observation_source_value"].fillna("").str.strip().str.upper()
    keep = code_value.isin(TINNITUS_OBSERVATION_CODES)
    if keep.any():
        selected = chunk.loc[keep].copy()
        selected["icd_code"] = code_value.loc[keep]
        observation_parts.append(selected)
    if chunk_number % 5 == 0:
        print(f"Observation chunks processed: {chunk_number}")

standard_tinnitus = pd.concat(observation_parts, ignore_index=True)
standard_tinnitus = standard_tinnitus.drop_duplicates("observation_id")
standard_tinnitus["person_id"] = standard_tinnitus["person_id"].str.strip()
standard_tinnitus["observation_date"] = pd.to_datetime(
    standard_tinnitus["observation_date"], errors="coerce"
).dt.date
standard_tinnitus["source_concept_id"] = pd.to_numeric(
    standard_tinnitus["observation_source_concept_id"], errors="coerce"
).astype("Int64")

print("Standard-tinnitus observation events:", f"{len(standard_tinnitus):,}")
display(
    standard_tinnitus.groupby("icd_code")
    .agg(
        n_events=("observation_id", "nunique"),
        n_people=("person_id", "nunique"),
    )
    .reset_index()
    .sort_values("n_events", ascending=False)
)

Observation chunks processed: 5
Standard-tinnitus observation events: 25,094


,icd_code,n_events,n_people
6,H93.13,10424,2479
1,388.30,4929,1537
5,H93.12,3594,837
4,H93.11,3008,789
7,H93.19,2920,1193
2,388.31,154,78
0,388.3,61,61
3,388.32,4,4


## 4. Attach OMOP ICD vocabulary and descriptions

The concept table is scanned once for the source concepts present in either evidence table. If a source concept is unavailable, vocabulary is inferred from the explicitly curated ICD code set.


In [5]:
needed_concept_ids = set(condition_events["source_concept_id"].dropna().astype("int64"))
needed_concept_ids.update(
    standard_tinnitus["source_concept_id"].dropna().astype("int64")
)

concept_parts = []
for chunk in pd.read_csv(
    CONCEPT_FILE,
    sep="\t",
    usecols=["concept_id", "concept_code", "concept_name", "vocabulary_id"],
    chunksize=1_000_000,
    low_memory=False,
):
    concept_id = pd.to_numeric(chunk["concept_id"], errors="coerce")
    keep = concept_id.isin(needed_concept_ids)
    if keep.any():
        concept_parts.append(chunk.loc[keep].copy())

concept = pd.concat(concept_parts, ignore_index=True)
concept["concept_id"] = pd.to_numeric(concept["concept_id"], errors="coerce").astype("Int64")
concept = concept.rename(columns={
    "concept_id": "source_concept_id",
    "concept_code": "concept_icd_code",
    "concept_name": "icd_description",
})
concept = concept.drop_duplicates("source_concept_id")

condition_events = condition_events.merge(
    concept, on="source_concept_id", how="left", validate="many_to_one"
)
standard_tinnitus = standard_tinnitus.merge(
    concept, on="source_concept_id", how="left", validate="many_to_one"
)

condition_events["vocabulary_id"] = condition_events["vocabulary_id"].where(
    condition_events["vocabulary_id"].isin(["ICD9CM", "ICD10CM"]),
    np.where(condition_events["icd_code"].str.match(r"^[A-Z]", na=False), "ICD10CM", "ICD9CM"),
)
standard_tinnitus["vocabulary_id"] = np.where(
    standard_tinnitus["icd_code"].isin(TINNITUS_ICD9), "ICD9CM", "ICD10CM"
)

print("Condition vocabulary counts:")
display(condition_events["vocabulary_id"].value_counts(dropna=False))
print("Standard-tinnitus vocabulary counts:")
display(standard_tinnitus["vocabulary_id"].value_counts(dropna=False))

Condition vocabulary counts:


vocabulary_id
ICD10CM    231771
ICD9CM      61077
Name: count, dtype: int64

Standard-tinnitus vocabulary counts:


vocabulary_id
ICD10CM    19946
ICD9CM      5148
Name: count, dtype: int64

## 5. Build event-level hearing and tinnitus evidence

Each clinical event appears once. Standard tinnitus is assigned to the broad current PhecodeX tinnitus parent `SO_397`; pulsatile events retain their PMBB-provided hierarchy.


In [6]:
hearing_evidence = condition_events.loc[
    condition_events["is_hearing_impairment"],
    [
        "condition_occurrence_id", "person_id", "condition_start_date",
        "icd_code", "concept_icd_code", "vocabulary_id", "icd_description",
        "mapped_phecodes",
    ],
].copy()
hearing_evidence = hearing_evidence.rename(columns={
    "condition_occurrence_id": "source_event_id",
    "condition_start_date": "event_date",
})
hearing_evidence["source_table"] = "condition_occurrence"
hearing_evidence["phenotype"] = "hearing_impairment"

pulsatile_evidence = condition_events.loc[
    condition_events["is_pulsatile_tinnitus"]
    & condition_events["icd_code"].isin(TINNITUS_ICD10_PULSATILE),
    [
        "condition_occurrence_id", "person_id", "condition_start_date",
        "icd_code", "concept_icd_code", "vocabulary_id", "icd_description",
        "mapped_phecodes",
    ],
].copy()
pulsatile_evidence = pulsatile_evidence.rename(columns={
    "condition_occurrence_id": "source_event_id",
    "condition_start_date": "event_date",
})
pulsatile_evidence["source_table"] = "condition_occurrence"
pulsatile_evidence["phenotype"] = "tinnitus"

standard_evidence = standard_tinnitus[
    [
        "observation_id", "person_id", "observation_date", "icd_code",
        "concept_icd_code", "vocabulary_id", "icd_description",
    ]
].copy()
standard_evidence = standard_evidence.rename(columns={
    "observation_id": "source_event_id",
    "observation_date": "event_date",
})
standard_evidence["mapped_phecodes"] = "SO_397"
standard_evidence["source_table"] = "observation"
standard_evidence["phenotype"] = "tinnitus"

evidence_columns = [
    "source_event_id", "source_table", "person_id", "event_date",
    "icd_code", "concept_icd_code", "vocabulary_id", "icd_description",
    "mapped_phecodes", "phenotype",
]

hearing_evidence = hearing_evidence[evidence_columns]
tinnitus_evidence = pd.concat(
    [standard_evidence[evidence_columns], pulsatile_evidence[evidence_columns]],
    ignore_index=True,
)
evidence = pd.concat([hearing_evidence, tinnitus_evidence], ignore_index=True)

evidence["source_event_id"] = evidence["source_event_id"].astype("string")
evidence["event_key"] = evidence["source_table"] + ":" + evidence["source_event_id"]
evidence = evidence.drop_duplicates(["phenotype", "event_key"])

print("Hearing events:", f"{len(hearing_evidence):,}")
print("Tinnitus events:", f"{len(tinnitus_evidence):,}")
display(
    evidence.groupby(["phenotype", "source_table", "vocabulary_id"])
    .agg(n_events=("event_key", "nunique"), n_people=("person_id", "nunique"))
    .reset_index()
)

Hearing events: 134,917
Tinnitus events: 28,110


,phenotype,source_table,vocabulary_id,n_events,n_people
0,hearing_impairment,condition_occurrence,ICD10CM,107035,9142
1,hearing_impairment,condition_occurrence,ICD9CM,27882,3776
2,tinnitus,condition_occurrence,ICD10CM,3016,397
3,tinnitus,observation,ICD10CM,19946,4227
4,tinnitus,observation,ICD9CM,5148,1604


## 6. Assign cases, exclusions, and controls

The related-auditory exclusion set combines all `SO_390`–`SO_399` condition events with standard-tinnitus observation events. Target status takes priority over exclusion status.


In [7]:
covars = pd.read_csv(COV_FILE, sep="\t", low_memory=False)
covars.columns = covars.columns.str.strip()
covars["person_id"] = covars["person_id"].astype("string").str.strip()
covars = covars.drop_duplicates("person_id")

hearing_dates = (
    hearing_evidence[["person_id", "event_date"]]
    .dropna().drop_duplicates()
)
tinnitus_dates = (
    tinnitus_evidence[["person_id", "event_date"]]
    .dropna().drop_duplicates()
)

related_ear_dates = pd.concat(
    [
        ear_phe[["person_id", "condition_start_date"]].rename(
            columns={"condition_start_date": "event_date"}
        ),
        standard_evidence[["person_id", "event_date"]],
    ],
    ignore_index=True,
).dropna().drop_duplicates()

related_counts = (
    related_ear_dates.groupby("person_id")["event_date"]
    .nunique().rename("n_related_ear_dates").reset_index()
)


def derive_phenotype(target_dates, phenotype_name, target_phecode):
    target_counts = (
        target_dates.groupby("person_id")["event_date"]
        .agg(n_target_dates="nunique", first_date="min", last_date="max")
        .reset_index()
    )

    result = covars[["person_id"]].copy()
    result["phenotype"] = phenotype_name
    result["target_phecode"] = target_phecode
    result = result.merge(target_counts, on="person_id", how="left")
    result = result.merge(related_counts, on="person_id", how="left")
    result["n_target_dates"] = result["n_target_dates"].fillna(0).astype(int)
    result["n_related_ear_dates"] = result["n_related_ear_dates"].fillna(0).astype(int)

    result["status"] = np.select(
        [
            result["n_target_dates"] >= 2,
            result["n_target_dates"] == 1,
            result["n_related_ear_dates"] >= 1,
        ],
        ["case", "excluded_one_date", "excluded_related_ear_phenotype"],
        default="control",
    )
    return result


hearing = derive_phenotype(hearing_dates, "hearing_impairment", "SO_396")
tinnitus = derive_phenotype(tinnitus_dates, "tinnitus", "SO_397")
status = pd.concat([hearing, tinnitus], ignore_index=True)

summary = (
    status.groupby(["phenotype", "target_phecode", "status"])
    .size().rename("n_people").reset_index()
)
display(summary)

,phenotype,target_phecode,status,n_people
0,hearing_impairment,SO_396,case,6752
1,hearing_impairment,SO_396,control,50755
2,hearing_impairment,SO_396,excluded_one_date,4007
3,hearing_impairment,SO_396,excluded_related_ear_phenotype,9411
4,tinnitus,SO_397,case,2749
5,tinnitus,SO_397,control,50755
6,tinnitus,SO_397,excluded_one_date,2598
7,tinnitus,SO_397,excluded_related_ear_phenotype,14823


In [9]:
for event_table in [hearing_evidence, tinnitus_evidence]:
    event_table["source_event_id"] = (
        event_table["source_event_id"].astype("string")
    )
    event_table["event_key"] = (
        event_table["source_table"]
        + ":"
        + event_table["source_event_id"]
    )

hearing_evidence = (
    hearing_evidence
    .drop_duplicates("event_key")
    .copy()
)

tinnitus_evidence = (
    tinnitus_evidence
    .drop_duplicates("event_key")
    .copy()
)

## 7. Required quality-control checks

These assertions ensure that no one-date participant or related-auditory participant enters a control group.


In [10]:
assert (status.loc[status.status == "case", "n_target_dates"] >= 2).all()
assert (status.loc[status.status == "excluded_one_date", "n_target_dates"] == 1).all()
assert (status.loc[status.status == "control", "n_target_dates"] == 0).all()
assert (status.loc[status.status == "control", "n_related_ear_dates"] == 0).all()
assert not evidence.duplicated(["phenotype", "event_key"]).any()

for phenotype_name in TARGETS:
    phenotype_status = status[status["phenotype"] == phenotype_name]
    assert len(phenotype_status) == phenotype_status["person_id"].nunique()

expected_tinnitus_codes = (
    TINNITUS_ICD10_STANDARD | TINNITUS_ICD10_PULSATILE | TINNITUS_ICD9
)
observed_tinnitus_codes = set(tinnitus_evidence["icd_code"].dropna())
unexpected_tinnitus_codes = observed_tinnitus_codes - expected_tinnitus_codes
assert not unexpected_tinnitus_codes, unexpected_tinnitus_codes

print("Case/control and event-level QC passed.")
display(
    tinnitus_evidence.groupby(["source_table", "vocabulary_id", "icd_code"])
    .agg(n_events=("event_key", "nunique"), n_people=("person_id", "nunique"))
    .reset_index()
)

Case/control and event-level QC passed.


,source_table,vocabulary_id,icd_code,n_events,n_people
0,condition_occurrence,ICD10CM,H93.A1,688,116
1,condition_occurrence,ICD10CM,H93.A2,821,115
2,condition_occurrence,ICD10CM,H93.A3,633,99
3,condition_occurrence,ICD10CM,H93.A9,874,163
4,observation,ICD10CM,H93.11,3008,789
5,observation,ICD10CM,H93.12,3594,837
6,observation,ICD10CM,H93.13,10424,2479
7,observation,ICD10CM,H93.19,2920,1193
8,observation,ICD9CM,388.3,61,61
9,observation,ICD9CM,388.30,4929,1537


## 8. Export audit, SAIGE, and PLINK files

Excluded participants are omitted from genetic analysis. Hearing impairment and tinnitus are exported as separate binary outcomes after intersection with the genotype sample list.


In [11]:
covars["SEX"] = covars["sequenced_gender"].map({"Male": 1, "Female": 2})
covars = covars.rename(columns={"sample_age": "AGE", "batch": "Batch"})
pc_columns = [f"exome_PC{i}" for i in range(1, 7) if f"exome_PC{i}" in covars.columns]
cov_columns = ["person_id", "Batch", "AGE", "SEX"] + pc_columns

fam = pd.read_csv(FAM_FILE, sep=r"\s+", header=None, dtype=str)
fam.columns = ["FID", "IID", "Father", "Mother", "Sex", "Phenotype"]
fam["IID"] = fam["IID"].astype("string").str.strip()
genotyped_ids = set(fam["IID"].dropna())

status.to_csv(OUT / "PMBBv4_hearing_tinnitus_all_statuses.csv.gz", index=False)
summary.to_csv(OUT / "PMBBv4_hearing_tinnitus_summary.csv", index=False)
evidence.to_csv(OUT / "PMBBv4_hearing_tinnitus_ICD_PhecodeX_evidence.csv.gz", index=False)

for phenotype_name in TARGETS:
    analysis = status[
        (status["phenotype"] == phenotype_name)
        & status["status"].isin(["case", "control"])
    ].copy()
    analysis["PHENO"] = (analysis["status"] == "case").astype(int)
    analysis = analysis.merge(covars[cov_columns], on="person_id", how="inner")

    eligible_before_fam = len(analysis)
    analysis = analysis[analysis["person_id"].isin(genotyped_ids)].copy()
    analysis = analysis.rename(columns={"person_id": "IID"})

    saige_columns = ["IID", "PHENO", "AGE", "SEX", "Batch"] + pc_columns
    saige = analysis[saige_columns].copy()
    saige.to_csv(OUT / f"{phenotype_name}_PMBBv4_SAIGE.txt", sep="\t", index=False)

    keep = saige[["IID"]].copy()
    keep.insert(0, "FID", keep["IID"])
    keep.to_csv(
        OUT / f"{phenotype_name}_PMBBv4_PLINK.keep",
        sep="\t", index=False, header=False,
    )

    print(f"\n{phenotype_name}")
    print("Eligible before genotype intersection:", eligible_before_fam)
    print("Genotyped:", len(saige))
    print("Cases:", int((saige.PHENO == 1).sum()))
    print("Controls:", int((saige.PHENO == 0).sum()))
    print("Missing covariates:")
    print(saige[["AGE", "SEX", "Batch"] + pc_columns].isna().sum())

print("\nSaved files:")
for path in sorted(OUT.iterdir()):
    print(path)


hearing_impairment
Eligible before genotype intersection: 57507
Genotyped: 57080
Cases: 6712
Controls: 50368
Missing covariates:
AGE          4
SEX          0
Batch        0
exome_PC1    0
exome_PC2    0
exome_PC3    0
exome_PC4    0
exome_PC5    0
exome_PC6    0
dtype: int64

tinnitus
Eligible before genotype intersection: 53504
Genotyped: 53100
Cases: 2732
Controls: 50368
Missing covariates:
AGE          4
SEX          0
Batch        0
exome_PC1    0
exome_PC2    0
exome_PC3    0
exome_PC4    0
exome_PC5    0
exome_PC6    0
dtype: int64

Saved files:
/project/hall/analysis/hearing-loss-genomics/analysis/elena/rarevariantExWAS/PMBBv4_phecodex/PMBBv4_hearing_tinnitus_ICD_PhecodeX_evidence.csv.gz
/project/hall/analysis/hearing-loss-genomics/analysis/elena/rarevariantExWAS/PMBBv4_phecodex/PMBBv4_hearing_tinnitus_all_statuses.csv.gz
/project/hall/analysis/hearing-loss-genomics/analysis/elena/rarevariantExWAS/PMBBv4_phecodex/PMBBv4_hearing_tinnitus_summary.csv
/project/hall/analysis/heari

## Methods language

> PMBB version 4.0 ICD-9-CM and ICD-10-CM diagnoses were used to construct separate hearing-impairment and tinnitus phenotypes. Hearing impairment was defined from condition events mapped by PMBB to PhecodeX SO_396 and its descendants. Because of OMOP table relocation in PMBB v4, standard tinnitus diagnoses (ICD-9-CM 388.3, 388.30, 388.31, and 388.32; ICD-10-CM H93.1, H93.11, H93.12, H93.13, and H93.19) were obtained from the observation table and assigned to the broad PhecodeX tinnitus phenotype SO_397. Pulsatile tinnitus diagnoses (ICD-10-CM H93.A, H93.A1, H93.A2, H93.A3, and H93.A9) were obtained from condition events and retained their PMBB PhecodeX hierarchy SO_397/SO_397.1/SO_397.11. Cases required qualifying evidence on at least two distinct dates. Participants with qualifying evidence on exactly one date were excluded. Controls had no target evidence and no related auditory PhecodeX ear-family evidence. No audiometric measurements were available or used.
